# When Accuracy Lies

**Notebook 1 of 5.** So far one number has judged every model: accuracy, the share of
predictions that are right. This notebook shows a problem where that number stops meaning
anything: **finding the few fraudulent card payments among many honest ones.**

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

## 1. The data

Card payments by European card holders over two days. The column `fraud` is 1 for a
fraudulent payment and 0 for an honest one. To protect the card holders, the original
columns were turned into anonymous numbers, `V1` to `V28` (25 of them are included here);
only `amount` is left as it was.

The full dataset has 284,807 payments. This file keeps all 492 frauds and a random sample
of the honest payments, 50,000 rows in all, so that it stays small enough to work with.

In [ ]:
df = pd.read_csv("../data/transactions.csv")
df.head()

In [ ]:
df["fraud"].value_counts()

**492 frauds, about 1 payment in 100.** When one class is this much rarer than the other,
the data is called **imbalanced**. The rare class, the one we care about, is the
**minority** class; here it is the positive class, 1.

## 2. A model that does nothing

The split first, stratified, as always. With so few frauds, stratifying matters: it
guarantees the test set gets its fair share of them.

In [ ]:
X = df.drop(columns="fraud")
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("frauds in the test set:", y_test.sum(), "of", len(y_test))

The majority baseline: call every payment honest.

In [ ]:
y_pred_baseline = np.zeros(len(y_test), dtype=int)   # 0 = honest, for every payment

print(f"accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")

**99 percent accurate, and it has never caught a single fraud.** A bank using it would
lose every fraudulent payment, and its accuracy report would look excellent.

## 3. A real model

> **Exercise:**
>
> Scale the features with a `StandardScaler` fitted on the training set, train a
> `LogisticRegression(max_iter=1000)`, and compute its test accuracy. How much better than
> the baseline is it?

<details><summary>
Click here for a hint...
</summary>

The same steps as in the logistic regression repository: `fit_transform` on the training
set, `transform` on the test set, then `fit` and `predict`. `max_iter=1000` only gives the
fitting more steps to finish; the default is too few for 26 columns.
</details>

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print(f"accuracy: {accuracy_score(y_test, y_pred):.4f}")

<details><summary>
Click here to compare your answer...
</summary>

About **99.8 percent**, less than one point above doing nothing. Judged by accuracy, the
model has barely earned its keep.
</details>

> **Exercise:**
>
> Accuracy hides the question that matters: **how many of the frauds were caught?** Count,
> for the baseline and for the model, the test frauds that were predicted as fraud.

<details><summary>
Click here for a hint...
</summary>

A fraud is caught where the truth is 1 **and** the prediction is 1:
`((y_test == 1) & (y_pred == 1)).sum()`.
</details>

In [ ]:
caught_baseline = ((y_test == 1) & (y_pred_baseline == 1)).sum()
caught_model = ((y_test == 1) & (y_pred == 1)).sum()

print(f"baseline caught {caught_baseline} of {y_test.sum()} frauds")
print(f"model caught    {caught_model} of {y_test.sum()} frauds")

<details><summary>
Click here to compare your answer...
</summary>

The baseline catches **0 of 123**; the model catches **97**. Accuracy put the two within a
point of each other, while one of them is useless and the other catches four frauds in
five. Accuracy counts every payment equally, and 99 in 100 payments are honest, so it
mostly measures how well a model recognizes honest payments.
</details>

## 4. Imbalance is everywhere

Fraud is not an unusual case. The events most worth predicting are often the rare ones:

- **fraud**, among millions of honest payments
- **a rare disease**, among many healthy patients
- **a defective part**, on a production line that mostly makes good ones

The same thing hurts every time. A model learns by getting as many training rows right as
it can, and when 99 rows in 100 belong to one class, ignoring the other class costs almost
nothing. So models lean toward the majority, and accuracy does not notice.

Two things follow, and they are the rest of this repository:

1. **Better metrics**, which look at each class separately: notebooks 02 to 04.
2. **Ways to handle the imbalance** itself: notebook 05.

## Summary

- On **imbalanced** data, a model that always predicts the majority class gets a high
  accuracy and is useless.
- Accuracy counts every row equally, so it is dominated by the majority class.
- The question that matters, **how many of the rare cases were found**, needs a different
  kind of metric.

Next, in `02_confusion_matrix.ipynb`: the table every classification metric is built from.